In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone your GitHub repo
!git clone https://github.com/emmadionne1/Multimodal-Machine-Learning-Project.git
%cd Multimodal-Machine-Learning-Project


Cloning into 'Multimodal-Machine-Learning-Project'...
remote: Enumerating objects: 196, done.
remote: Total 196 (delta 0), reused 0 (delta 0), pack-reused 196 (from 1)
Receiving objects: 100% (196/196), 862.84 MiB | 40.03 MiB/s, done.
Resolving deltas: 100% (85/85), done.
Updating files: 100% (98/98), done.
/content/Multimodal-Machine-Learning-Project


In [ ]:
!pip install torch transformers datasets accelerate timm pillow tqdm numpy sentencepiece evaluate config

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


In [ ]:
%cd Multimodal-Machine-Learning-Project


/content/Multimodal-Machine-Learning-Project


In [ ]:

import torch
import re
import numpy as np
from tqdm import tqdm
from PIL import Image
from datasets import load_dataset

from base_architecture import (
    load_vision_model, load_language_model,
    Qwen3VLProjector, CustomVLM, DEVICE
)


DEVICE = "cuda"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
MODEL_VISION = "google/siglip2-so400m-patch16-512"
MODEL_LLM = "Qwen/Qwen3-4B-Instruct-2507"

PROJECTOR_PATH = "/content/drive/MyDrive/Multimodal Project/pretraining weights/7e3_a80_mediumishish_batch.pt"
#"/content/Multimodal-Machine-Learning-Project/outputs/final_run_full_dataset_A40_proj_final_weights.pt"  # 🔥 UPLOAD YOUR .pt FILE HERE

DATASET_NAME = "chiragtubakad/chart-to-table"
MAX_NEW_TOKENS = 512

PROMPT = "Extract all the information and convert this chart image into a Markdown table. Output only the table."

In [ ]:
# Load vision model
vision_model, processor = load_vision_model(MODEL_VISION)
v_conf = vision_model.config.vision_config if hasattr(vision_model.config, "vision_config") else vision_model.config
vision_dim = v_conf.hidden_size

# Load language model
language_model, tokenizer = load_language_model(MODEL_LLM)
llm_dim = language_model.config.hidden_size

# Add image token if missing
if "<image>" not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
    language_model.resize_token_embeddings(len(tokenizer))

image_token_id = tokenizer.convert_tokens_to_ids("<image>")

# 🔥 Load YOUR projector
projector = Qwen3VLProjector(vision_dim, llm_dim, 2).to(DEVICE, dtype=torch.bfloat16)
projector.load_state_dict(torch.load(PROJECTOR_PATH, map_location="cpu"))

vision_model = vision_model.to(DEVICE, dtype=torch.bfloat16)
language_model = language_model.to(DEVICE, dtype=torch.bfloat16)
projector = projector.to(DEVICE, dtype=torch.bfloat16)

# Build full model
model = CustomVLM(
    vision_model,
    processor,
    language_model,
    tokenizer,
    projector,
    image_token_id
)
model = model.to(DEVICE)
model.eval()
print("✅ Model loaded successfully")

Loading Vision: google/siglip2-so400m-patch16-512


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✅ Model loaded successfully


In [ ]:
ds = load_dataset(DATASET_NAME)["train"]
print("Dataset size:", len(ds))

README.md:   0%|          | 0.00/481 [00:00<?, ?B/s]

data/train-00000-of-00001-8f3122c12de7e5(…):   0%|          | 0.00/33.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/990 [00:00<?, ? examples/s]

Dataset size: 990


In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s\.]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def relaxed_match(pred, gt):
    pred_norm = normalize_text(pred)
    gt_norm = normalize_text(gt)

    if pred_norm == gt_norm or gt_norm in pred_norm:
        return True

    try:
        pv = float(re.findall(r"-?\d+\.?\d*", pred)[0])
        gv = float(re.findall(r"-?\d+\.?\d*", gt)[0])
        if abs(pv - gv) / max(abs(gv), 1e-6) <= 0.05:
            return True
    except:
        pass

    return False

def f1_score(pred_list, gt_list):
    matched_gt = set()
    matched_pred = set()

    for i, p in enumerate(pred_list):
        for j, g in enumerate(gt_list):
            if j in matched_gt:
                continue
            if relaxed_match(p, g):
                matched_pred.add(i)
                matched_gt.add(j)
                break

    precision = len(matched_pred) / max(len(pred_list), 1)
    recall = len(matched_gt) / max(len(gt_list), 1)
    return 2 * precision * recall / max(precision + recall, 1e-6)

def levenshtein(a, b):
    a, b = str(a), str(b)
    if len(a) < len(b): a, b = b, a
    if not b: return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1]+1, prev[j]+1, prev[j-1]+(ca!=cb)))
        prev = cur
    return prev[-1]

In [ ]:
exact_correct = 0
relaxed_correct = 0
rms_f1_list = []
lev_list = []
total = 0

# limit first for safety
ds_eval = ds#.select(range(50))  # 🔥 increase later

for sample in tqdm(ds_eval):

    # ---- Prompt ----
    messages = [{"role": "user", "content": "<image>\n" + PROMPT}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # ---- Image ----
    image = sample["image"]
    if image.mode != "RGB":
        image = image.convert("RGB")

    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(DEVICE, dtype=torch.bfloat16)

    # ---- Text ----
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    attention_mask = torch.ones_like(inputs.input_ids).to(DEVICE)

    # ---- Generate ----
    with torch.no_grad():
        pred = model.generate_text(
            input_ids=inputs.input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            max_new_tokens=MAX_NEW_TOKENS
        )

    pred = pred.strip()
    gt = sample["text"]

    # ---- Metrics ----
    exact = normalize_text(pred) == normalize_text(gt)
    relaxed = relaxed_match(pred, gt)

    if exact: exact_correct += 1
    if relaxed: relaxed_correct += 1

    pred_cells = re.split(r"[\n,\|]+", pred)
    gt_cells = re.split(r"[\n,\|]+", gt)

    f1 = f1_score(
        [c for c in pred_cells if c.strip()],
        [c for c in gt_cells if c.strip()]
    )
    rms_f1_list.append(f1)

    lev = levenshtein(normalize_text(pred), normalize_text(gt))
    lev_list.append(lev)

    total += 1

    # print a few samples
    if total <= 3:
        print("\n=== SAMPLE ===")
        print("PRED:\n", pred)
        print("GT:\n", gt)
        print(f"F1: {f1:.4f} | Lev: {lev}")

print("\n=== FINAL RESULTS ===")
print(f"Total: {total}")
print(f"Exact Acc: {exact_correct/total:.4f}")
print(f"Relaxed Acc: {relaxed_correct/total:.4f}")
print(f"RMS F1: {np.mean(rms_f1_list):.4f}")
print(f"Mean Levenshtein: {np.mean(lev_list):.4f}")


  0%|          | 0/990 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

  0%|          | 1/990 [00:03<53:07,  3.22s/it]


=== SAMPLE ===
PRED:
 a graph showing the relationship between two variables
GT:
 | Category   |         X |       Y |
| SbST       |  78.2846  | 92.0597 |
| SbST       |  80.9146  | 72.2484 |
| SbST       |  94.0858  | 84.9057 |
| SbST       |  97.5217  | 86.4296 |
| SbST       |  97.0339  | 89.4987 |
| SbST       |  94.4888  | 89.5198 |
| SbST       |  96.8642  | 92.1232 |
| SbST       |  98.2853  | 94.7055 |
| SbST       | 100.958   | 91.3824 |
| SbST       |  98.1287  | 87.7248 |
| CladHC     |  44.0008  | 89.4352 |
| CladHC     |  60.7928  | 86.6322 |
| CladHC     |  65.8043  | 86.4175 |
| CladHC     |  70.3735  | 84.7363 |
| CladHC     |  71.7309  | 86.7683 |
| CladHC     |  71.9005  | 91.8481 |
| CladHC     |  82.6962  | 89.8162 |
| CladHC     |  15.992   | 92.1021 |
| CladHC     |   1.25137 | 94.1763 |
| CladHC     |  32.5652  | 72.7286 |
F1: 0.0000 | Lev: 434



  0%|          | 2/990 [00:04<31:58,  1.94s/it]


=== SAMPLE ===
PRED:
 a line graph showing the number of people who have been affected by the disease
GT:
 | Category                 |            X |          Y |
| [unnamed data series #0] | -0.00132895  | 0.0066229  |
| [unnamed data series #0] |  0.00283124  | 0.0305805  |
| [unnamed data series #0] |  0.0132317   | 0.0121516  |
| [unnamed data series #0] |  0.0208588   | 0.0066229  |
| [unnamed data series #0] |  0.0193565   | 0.0239     |
| [unnamed data series #0] |  0.0257123   | 0.0356485  |
| [unnamed data series #0] |  0.0361128   | 0.0273554  |
| [unnamed data series #0] |  0.00837817  | 0.0632919  |
| [unnamed data series #0] |  0.00283124  | 0.108903   |
| [unnamed data series #0] |  0.00352461  | 0.122725   |
| [unnamed data series #0] |  0.0104583   | 0.172483   |
| [unnamed data series #0] |  0.0264057   | 0.126872   |
| [unnamed data series #0] |  0.0333394   | 0.0757314  |
| [unnamed data series #0] |  0.0533314   | 0.031502   |
| [unnamed data series #0] |  0.06696


  0%|          | 3/990 [00:05<23:55,  1.45s/it]


=== SAMPLE ===
PRED:
 a scatter plot showing the relationship between two variables
GT:
 | Category                       |         X |       Y |
| Expression down                | -1.68313  | 2.06587 |
| Expression down                | -1.50823  | 2.56886 |
| Expression down                | -0.808642 | 3.39521 |
| Expression down                | -0.592593 | 2.47305 |
| Expression down                | -1.12757  | 2.54491 |
| Expression down                | -1.03498  | 2.66467 |
| Expression down                | -1.107    | 2.13772 |
| Expression down                | -0.839506 | 2.11377 |
| Expression down                | -0.633745 | 2.23353 |
| Expression down                | -0.510288 | 2.22156 |
| On Chromosome IV               | -1.59053  | 6.29102 |
| On Chromosome IV               | -1.8786   | 3.51257 |
| On Chromosome IV               | -1.54938  | 3.78084 |
| On Chromosome IV               | -0.792181 | 3.74251 |
| On Chromosome IV               | -0.528807 | 3.17725 


100%|██████████| 990/990 [1:15:04<00:00,  4.55s/it]


=== FINAL RESULTS ===
Total: 990
Exact Acc: 0.0000
Relaxed Acc: 0.0051
RMS F1: 0.0116
Mean Levenshtein: 1052.1859
